In [44]:
# ==============================
# IMPORT LIBRARIES
# ==============================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings("ignore")

In [ ]:
train = pd.read_csv("TRAIN.csv")

train.head()

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
0,0.185570,0.004568,0.005362,0.003335,0.005415,0.004895,0.012764,0.120138,0.140450,3.361753,...,0.041526,-0.230857,0.003310,0.042250,0.005975,0.002104,0.013878,0.001518,0.011347,0
1,0.369536,0.003983,0.003386,0.004902,0.007570,0.012136,0.118050,0.323925,0.132093,2.766117,...,-0.141285,-6.222857,0.834177,0.227968,0.018463,-0.020487,0.001246,0.007489,0.008724,1
2,0.602510,0.008442,0.012961,0.012870,0.046885,0.115401,0.065688,0.306677,0.498805,4.521201,...,0.011334,10.335251,-0.276614,-0.198900,-0.012756,0.014286,-0.001866,0.002687,0.013452,1
3,0.347957,0.064721,0.013611,0.011541,0.006492,0.008690,0.013192,0.164553,0.298665,3.170847,...,0.190479,2.864912,-1.921939,0.891690,1.108098,0.635431,0.081368,-0.000225,0.009166,0
4,0.233653,0.012217,0.010088,0.022095,0.026040,0.015062,0.016063,0.084648,0.213367,8.150943,...,0.203164,0.001812,-0.092731,0.005280,-0.213985,0.032195,0.002081,0.028930,-0.025912,1


In [ ]:
train.shape
train.info()
train.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43776 entries, 0 to 43775
Data columns (total 48 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   F01     43776 non-null  float64
 1   F02     43776 non-null  float64
 2   F03     43776 non-null  float64
 3   F04     43776 non-null  float64
 4   F05     43776 non-null  float64
 5   F06     43776 non-null  float64
 6   F07     43776 non-null  float64
 7   F08     43776 non-null  float64
 8   F09     43776 non-null  float64
 9   F10     43776 non-null  float64
 10  F11     43776 non-null  float64
 11  F12     43776 non-null  float64
 12  F13     43776 non-null  float64
 13  F14     43776 non-null  float64
 14  F15     43776 non-null  float64
 15  F16     43776 non-null  float64
 16  F17     43776 non-null  float64
 17  F18     43776 non-null  float64
 18  F19     43776 non-null  float64
 19  F20     43776 non-null  float64
 20  F21     43776 non-null  float64
 21  F22     43776 non-null  float64
 22

,F01,F02,F03,F04,F05,F06,F07,F08,F09,F10,...,F39,F40,F41,F42,F43,F44,F45,F46,F47,Class
count,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,...,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000,43776.000000
mean,0.567525,0.014285,0.015551,0.026779,0.087637,0.155142,0.150656,0.255644,0.361146,3.762811,...,0.023477,-0.039795,0.005846,-0.000089,0.003215,0.001007,0.004599,0.003981,-0.006780,0.395445
std,0.738884,0.019607,0.022504,0.050725,0.188394,0.377655,0.339948,0.354816,0.440134,1.271343,...,0.707224,8.462749,1.631328,1.103486,0.354742,0.171935,0.201611,0.202130,0.562031,0.488952
min,0.100025,0.003001,0.000752,0.000682,0.000835,0.000854,0.000702,0.001399,0.001164,0.038349,...,-22.647806,-28.026880,-9.109769,-9.275399,-10.607320,-5.910117,-11.029342,-18.853842,-44.878769,0.000000
25%,0.198843,0.004778,0.005437,0.006290,0.012050,0.014112,0.015084,0.072098,0.128780,3.192588,...,-0.036589,-0.254039,-0.117789,-0.015759,-0.011839,-0.007214,-0.005226,-0.000826,-0.008882,0.000000
50%,0.288778,0.007988,0.008905,0.012619,0.023681,0.032396,0.029157,0.119095,0.214665,3.604579,...,-0.000210,-0.000757,-0.002891,-0.000111,-0.000045,-0.000170,0.000295,0.000092,-0.000083,0.000000
75%,0.528760,0.014382,0.016610,0.027599,0.063469,0.115950,0.116992,0.265227,0.381230,4.035824,...,0.043276,0.171122,0.107351,0.013399,0.009806,0.005892,0.006058,0.002405,0.007418,1.000000
max,12.779628,0.199925,0.506419,0.851009,5.017180,7.249545,6.556998,5.960009,5.085546,50.624777,...,89.378571,28.471415,10.907778,9.374269,11.399358,9.139964,18.623611,8.960172,40.818882,1.000000


In [ ]:
# ==============================
# CHECK TARGET DISTRIBUTION
# ==============================

print(train['Class'].value_counts())
print(train['Class'].value_counts(normalize=True))

Class
0    26465
1    17311
Name: count, dtype: int64
Class
0    0.604555
1    0.395445
Name: proportion, dtype: float64


In [ ]:
## can  drop  features  with  corr near 0
# ==============================
# CORRELATION WITH TARGET
# ==============================

corr = train.corr()['Class'].sort_values(ascending=False)

print("Top 10 Positive Correlations:")
print(corr.head(10))

print("\nTop 10 Negative Correlations:")
print(corr.tail(10))



Top 10 Positive Correlations:
Class    1.000000
F01      0.386981
F09      0.377363
F29      0.364944
F19      0.358187
F21      0.344699
F05      0.342236
F25      0.338634
F07      0.334932
F27      0.334544
Name: Class, dtype: float64

Top 10 Negative Correlations:
F45    0.004567
F38    0.004186
F40    0.003154
F41   -0.000028
F20   -0.004982
F35   -0.005728
F42   -0.007027
F43   -0.008375
F44   -0.015088
F34   -0.024363
Name: Class, dtype: float64


In [ ]:
# ==============================
# FEATURE & TARGET SEPARATION
# ==============================

X = train.drop("Class", axis=1)
y = train["Class"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (43776, 47)
Target shape: (43776,)


In [ ]:
X.shape
y.shape

(43776,)

In [ ]:
# ==============================
# STRATIFIED K-FOLD SETUP
# ==============================

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [43]:
# pip install xgboost

In [ ]:
print(X.columns)

Index(['F01', 'F02', 'F03', 'F04', 'F05', 'F06', 'F07', 'F08', 'F09', 'F10',
       'F11', 'F12', 'F13', 'F14', 'F15', 'F16', 'F17', 'F18', 'F19', 'F20',
       'F21', 'F22', 'F23', 'F24', 'F25', 'F26', 'F27', 'F28', 'F29', 'F30',
       'F31', 'F32', 'F33', 'F34', 'F35', 'F36', 'F37', 'F38', 'F39', 'F40',
       'F41', 'F42', 'F43', 'F44', 'F45', 'F46', 'F47'],
      dtype='object')


In [48]:
# ==============================
# FULL CV TRAINING
# ==============================

all_val_probs = []
all_val_true = []

f1_scores = []
auc_scores = []

for fold, (train_index, val_index) in enumerate(skf.split(X, y)):

    print(f"\n========== Fold {fold+1} ==========")

    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]

    # Handle imbalance inside fold
    neg = sum(y_train == 0)
    pos = sum(y_train == 1)
    scale_pos_weight = neg / pos

    model = XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss"
    )

    model.fit(X_train, y_train)

    val_probs = model.predict_proba(X_val)[:, 1]
    val_preds = (val_probs >= 0.5).astype(int)

    fold_f1 = f1_score(y_val, val_preds)
    fold_auc = roc_auc_score(y_val, val_probs)

    print("Fold F1:", fold_f1)
    print("Fold ROC-AUC:", fold_auc)

    f1_scores.append(fold_f1)
    auc_scores.append(fold_auc)

    all_val_probs.extend(val_probs)
    all_val_true.extend(y_val)


========== Fold 1 ==========
Fold F1: 0.9810007251631617
Fold ROC-AUC: 0.9988665910260524

========== Fold 2 ==========
Fold F1: 0.9806601715864476
Fold ROC-AUC: 0.9986268556303667

========== Fold 3 ==========
Fold F1: 0.9819399941741916
Fold ROC-AUC: 0.9986539779875605

========== Fold 4 ==========
Fold F1: 0.981301637918539
Fold ROC-AUC: 0.9984430566383579

========== Fold 5 ==========
Fold F1: 0.9798170466095543
Fold ROC-AUC: 0.998703638641577


In [49]:
# ==============================
# CV RESULTS
# ==============================

print("\n==============================")
print("Mean CV F1:", np.mean(f1_scores))
print("Mean CV ROC-AUC:", np.mean(auc_scores))
print("==============================")


Mean CV F1: 0.9809439150903788
Mean CV ROC-AUC: 0.9986588239847828


In [50]:
# ==============================
# GLOBAL THRESHOLD OPTIMIZATION
# ==============================

all_val_probs = np.array(all_val_probs)
all_val_true = np.array(all_val_true)

thresholds = np.arange(0.1, 0.9, 0.01)

best_threshold = 0.5
best_f1 = 0

for t in thresholds:
    preds = (all_val_probs >= t).astype(int)
    score = f1_score(all_val_true, preds)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Global Best Threshold:", best_threshold)
print("Global Best CV F1:", best_f1)

final_auc = roc_auc_score(all_val_true, all_val_probs)
print("Final CV ROC-AUC:", final_auc)

Global Best Threshold: 0.44999999999999984
Global Best CV F1: 0.9817581653926337
Final CV ROC-AUC: 0.998660142368543


In [51]:
# ==============================
# FINAL MODEL TRAINING (FULL DATA)
# ==============================

neg = sum(y == 0)
pos = sum(y == 1)
scale_pos_weight = neg / pos

final_model = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)

final_model.fit(X, y)

print("Final model trained on full data.")

Final model trained on full data.


In [ ]:
# ==============================
# TEST PREDICTION (CORRECT FORMAT)
# ==============================

test = pd.read_csv("TEST.csv")

# Save ID column
test_ids = test["ID"]

# Drop ID before prediction
test_features = test.drop("ID", axis=1)

# Predict probabilities
test_probs = final_model.predict_proba(test_features)[:, 1]

# Use best threshold found from CV (example: 0.4)
test_preds = (test_probs >= best_threshold).astype(int)

# Create submission in exact required format
submission = pd.DataFrame({
    "ID": test_ids,
    "CLASS": test_preds
})

# Ensure same order as TEST.csv (already preserved)
submission.to_csv("FINAL.csv", index=False)

print("FINAL.csv created successfully!")

FINAL.csv created successfully!
